# 📖 Notebook 4: Cache Patterns & Stampede Protection

Notebooks 1–3 answered *where* data lives and *how* it stays in sync across nodes.
This notebook answers the other half of the story:

> When the cache *misses*, how does the data get there — and what happens when
> **everyone misses at the same time**?

We'll cover the four classic **cache access patterns** (cache-aside, read-through,
write-through, write-behind), then tackle the most famous distributed-cache failure
mode: the **cache stampede** (a.k.a. thundering herd).

## Learning Objectives

By the end of this notebook, you'll understand:
- The four cache access patterns and when to use each
- Why naive cache-aside can serve stale data forever
- What a cache stampede is and how it takes down databases
- How to stop a stampede with a **Redis distributed lock** (SET NX)
- Why "negative caching" (caching the absence of a value) matters


## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/distributed-cache
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of the notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In this notebook we only need **one** Redis node as our cache — the goal is to
understand *patterns*, not partitioning. We'll use `node-1` as the cache, and
simulate a slow "database" with `time.sleep`.


In [ ]:
import redis
import time
import threading
import uuid
import queue

# Use one Redis node as the cache
cache = redis.Redis(host="localhost", port=6381, decode_responses=True)
cache.flushall()
cache.ping()
print("✅ cache is up (node-1 on port 6381)")


## 🗄️ The Fake Database

To see cache patterns in action, we need a "database" that's slow enough to
notice. This `FakeDB` sleeps 100ms on every read — just like a real SQL query
against a remote Postgres. It also **counts** every read so we can *prove* how
many times the DB was hit.


In [ ]:
class FakeDB:
    """A fake database with latency and a built-in hit counter."""

    def __init__(self, latency_ms: int = 100):
        self.data = {}
        self.latency_ms = latency_ms
        self.read_count = 0
        self.write_count = 0
        self._lock = threading.Lock()

    def read(self, key: str):
        time.sleep(self.latency_ms / 1000)
        with self._lock:
            self.read_count += 1
        return self.data.get(key)

    def write(self, key: str, value: str):
        time.sleep(self.latency_ms / 1000)
        with self._lock:
            self.write_count += 1
        self.data[key] = value

    def reset_counters(self):
        self.read_count = 0
        self.write_count = 0


db = FakeDB(latency_ms=100)

# Seed some initial data
db.data["product:42"] = "iPhone 15 — $999"
db.data["product:99"] = "AirPods Pro — $249"
db.reset_counters()

print(f"💾 DB seeded with {len(db.data)} rows, latency={db.latency_ms}ms")


---

## Part 1: Cache-Aside (Lazy Loading) — The Default Pattern

In **cache-aside**, the *application* owns the logic:

1. Check the cache.
2. On **hit** → return the value.
3. On **miss** → read from the DB, write it to the cache, return it.

This is by far the most common pattern. Redis + Python apps almost always use it.

```
  app ──► cache.get(key)
           │
      miss ▼
       db.read(key) ──► cache.set(key, value) ──► return value
```


### ❌ Bad: Cache-Aside Without a TTL or Invalidation

The naive version "just works" — until the DB changes. Then the cache serves
stale data **forever**.


In [ ]:
def cache_aside_bad_get(key: str) -> str:
    """Cache-aside with NO ttl and NO invalidation on writes. Will serve stale data."""
    cached = cache.get(key)
    if cached is not None:
        return cached
    value = db.read(key)
    if value is not None:
        cache.set(key, value)  # No TTL — lives forever
    return value


def cache_aside_bad_update(key: str, new_value: str):
    """Update the DB but forget to touch the cache."""
    db.write(key, new_value)
    # BUG: cache still has the old value


cache.flushall()
db.reset_counters()

# First read — MISS, hits DB
v1 = cache_aside_bad_get("product:42")
print(f"1. First read  → {v1}  (db reads so far: {db.read_count})")

# Update the underlying data
cache_aside_bad_update("product:42", "iPhone 15 — $899 (SALE)")
print(f"2. DB updated to: {db.data['product:42']}")

# Read again — cache returns the STALE old value
v2 = cache_aside_bad_get("product:42")
print(f"3. Second read → {v2}  ❌ STALE — cache never got the memo")


### ✅ Good: Cache-Aside with TTL + Invalidate-on-Write

Two small fixes eliminate 99% of staleness problems:

1. **TTL**: every cache entry expires after N seconds → worst-case staleness is bounded.
2. **Invalidate-on-write**: when the DB changes, delete the cache key so the next
   read repopulates it fresh.

The TTL acts as a *safety net* even if invalidation is missed (e.g. a bug, a
crashed process, or a write from outside your app).


In [ ]:
CACHE_TTL = 60  # seconds


def cache_aside_good_get(key: str) -> str:
    cached = cache.get(key)
    if cached is not None:
        return cached
    value = db.read(key)
    if value is not None:
        cache.set(key, value, ex=CACHE_TTL)  # TTL safety net
    return value


def cache_aside_good_update(key: str, new_value: str):
    db.write(key, new_value)
    cache.delete(key)  # Invalidate → next read will repopulate fresh


cache.flushall()
db.reset_counters()

# Warm the cache
cache_aside_good_get("product:42")

# Update the DB and invalidate
cache_aside_good_update("product:42", "iPhone 15 — $899 (SALE)")

# Next read — MISS (because we invalidated), then fresh value
v = cache_aside_good_get("product:42")
print(f"After invalidate-on-write → {v}  ✅ fresh")
print(f"Total DB reads: {db.read_count} (two: initial warm + repopulate after invalidate)")


---

## Part 2: Read-Through — Let the Cache Own the Lookup

**Cache-aside** puts the "on miss → read DB" logic in the *application*.
**Read-through** moves that logic *into the cache itself* (or a library wrapper).

```
  app ──► cache.get(key)   # that's it
             │
      miss ▼ (transparent to app)
           db.read(key) ──► populate cache ──► return
```

The app only ever calls `cache.get(...)`. Cleaner API, but the wrapper has to
know how to query the DB. Redis itself doesn't do this natively, but client
libraries (e.g. Spring Cache, Django's `cache.get_or_set`) implement it on top.


In [ ]:
class ReadThroughCache:
    """Wraps a cache + DB so callers only see cache.get()."""

    def __init__(self, cache_client, db, ttl: int = 60):
        self.cache = cache_client
        self.db = db
        self.ttl = ttl

    def get(self, key: str):
        cached = self.cache.get(key)
        if cached is not None:
            return cached
        # Miss — the wrapper (not the app) calls the DB
        value = self.db.read(key)
        if value is not None:
            self.cache.set(key, value, ex=self.ttl)
        return value


rt = ReadThroughCache(cache, db, ttl=60)

cache.flushall()
db.reset_counters()

# App only ever touches the wrapper
print(rt.get("product:42"))      # miss → DB
print(rt.get("product:42"))      # hit  → cache
print(f"DB reads: {db.read_count} (should be 1)")


---

## Part 3: Write-Through — Keep Cache & DB in Lockstep on Writes

In **write-through**, every write goes to **both** the cache and the DB, synchronously,
before returning to the client.

```
  app ──► write(key, v) ──► cache.set(key, v) ──► db.write(key, v) ──► return
```

✅ Cache is always fresh — no invalidation needed.
❌ Every write pays for two hops (cache + DB latency).
❌ Failure semantics are tricky (see below).

### ⚠️ Failure modes to understand

If the **cache write succeeds but the DB write fails** → you have a cache entry for
data that **doesn't exist in the DB**. Solution: write to the DB **first**, then
update the cache. If the DB write succeeds but the cache write fails → next
read is a miss and repopulates. Less bad.


In [ ]:
def write_through(key: str, value: str):
    """DB first, then cache. On DB failure, cache is untouched (safer)."""
    db.write(key, value)          # source of truth first
    cache.set(key, value, ex=60)  # then the cache


cache.flushall()
db.reset_counters()

write_through("product:100", "Mug — $15")

# Read should be a cache HIT immediately
print(f"cache: {cache.get('product:100')}")
print(f"db:    {db.data['product:100']}")
print(f"db reads since write-through: {db.read_count} (should be 0 — no read needed)")


---

## Part 4: Write-Behind (Write-Back) — Speed at the Cost of Durability

**Write-behind** writes to the cache *immediately* and queues the DB write to
happen **asynchronously** in the background.

```
  app ──► cache.set(key, v) ──► return (fast!)
                              └── queue ──► db.write(key, v)   (later)
```

✅ Writes feel instant to the app.
❌ If the cache / app crashes before the queue drains → **writes are lost**.
❌ Readers going directly to the DB see stale data during the lag window.

This is how some NoSQL systems and the OS page cache work. It's powerful but
requires durable queues (e.g. Kafka, Redis Streams) to be safe in production.


In [ ]:
write_queue: "queue.Queue[tuple[str, str]]" = queue.Queue()
stop_flag = threading.Event()


def writer_worker():
    """Background worker that drains the queue into the DB."""
    while not stop_flag.is_set() or not write_queue.empty():
        try:
            key, value = write_queue.get(timeout=0.1)
        except queue.Empty:
            continue
        db.write(key, value)  # 100ms latency — but off the hot path
        write_queue.task_done()


def write_behind(key: str, value: str):
    cache.set(key, value, ex=60)
    write_queue.put((key, value))


# Start the background writer
worker = threading.Thread(target=writer_worker, daemon=True)
worker.start()

cache.flushall()
db.reset_counters()

# Fire off 5 writes and measure how fast the app sees them "complete"
start = time.time()
for i in range(5):
    write_behind(f"order:{i}", f"order data {i}")
app_elapsed = (time.time() - start) * 1000

print(f"App-visible write time for 5 writes: {app_elapsed:.1f} ms (compare to 5×100ms sync)")

# Immediately after, the DB hasn't caught up yet
print(f"DB writes right now:        {db.write_count} (likely 0 or very few)")
print(f"Queue depth right now:      {write_queue.qsize()}")

# Wait for the queue to drain
write_queue.join()
print(f"DB writes after drain:      {db.write_count} (should be 5)")

stop_flag.set()
worker.join(timeout=2)


> 💡 **Durability warning.** In our demo the queue is just an in-process
> `queue.Queue`. If the process crashes between `write_behind(...)` returning
> and the worker flushing, those writes are gone. Real systems back the queue
> with a durable log (Redis Streams, Kafka, a WAL on disk).


---

## Part 5: The Cache Stampede (Thundering Herd)

A **cache stampede** happens when a *hot* key expires (or was never populated)
and **many concurrent requests miss at the same time**. Every one of them
thinks *"I'll just read the DB and repopulate"*. The DB gets hammered by N
identical queries instead of 1.

This is arguably the #1 reason cache layers take down databases in production.

### Why is it "thundering"?

Imagine 500 web servers, each handling 200 requests/second, all serving a page
that reads `product:iphone`. That key's TTL expires. Suddenly **100,000 requests
per second** all miss and slam the DB with the same query.

### Let's prove it — deterministically

To make the demo reproducible, we'll use a `threading.Barrier` so all workers
line up and start **at exactly the same instant**.


In [ ]:
HOT_KEY = "product:iphone"
NUM_WORKERS = 20

# Make sure the hot key actually exists in the DB so a successful fetch
# will populate the cache — otherwise None is returned and every worker misses.
db.data[HOT_KEY] = "iPhone 15 Pro Max — $1199"


def naive_get(key: str) -> str:
    """Plain cache-aside — no stampede protection."""
    cached = cache.get(key)
    if cached is not None:
        return cached
    # MISS — fetch from DB and populate
    value = db.read(key)
    cache.set(key, value, ex=60)
    return value


def run_stampede(fetcher, label: str):
    cache.flushall()
    db.reset_counters()
    barrier = threading.Barrier(NUM_WORKERS)
    results = []

    def worker():
        barrier.wait()  # synchronize all workers
        results.append(fetcher(HOT_KEY))

    threads = [threading.Thread(target=worker) for _ in range(NUM_WORKERS)]
    start = time.time()
    for t in threads: t.start()
    for t in threads: t.join()
    elapsed = (time.time() - start) * 1000

    print(f"[{label}]")
    print(f"  workers: {NUM_WORKERS}")
    print(f"  DB reads triggered: {db.read_count}  {'🔥 STAMPEDE!' if db.read_count > 1 else '✅'}")
    print(f"  total elapsed: {elapsed:.0f} ms")
    print()


run_stampede(naive_get, "naive cache-aside")


You should see all 20 workers hit the DB — that's **20× the load** the DB
should have seen. In production with 10,000 workers, that's 10,000× the load
on a single query, frequently enough to cause an outage.

### 🛡️ Fix: Single-Flight with a Redis Distributed Lock

The idea: when a miss happens, only **one** worker is allowed to talk to the
DB. The others wait briefly and retry the cache.

We implement this with Redis's atomic `SET key value NX EX ttl`:

- `NX` → set **only if** the key doesn't exist (atomic "acquire lock").
- `EX ttl` → the lock **auto-expires** if the holder crashes (no dead locks).
- We store a random **token** as the value so only the holder can release it.
- Release uses a tiny **Lua script** for atomic compare-and-delete — otherwise
  a slow worker could delete a later worker's lock.


In [ ]:
# Lua script: delete the lock only if the token matches.
# Runs atomically inside Redis — no TOCTOU race.
RELEASE_LOCK_LUA = """
if redis.call('GET', KEYS[1]) == ARGV[1] then
    return redis.call('DEL', KEYS[1])
else
    return 0
end
"""
release_lock = cache.register_script(RELEASE_LOCK_LUA)


def single_flight_get(key: str, lock_ttl_ms: int = 5000, wait_ms: int = 50) -> str:
    """Cache-aside with a distributed lock so only one worker recomputes a miss."""
    # 1. Fast path — cache hit
    cached = cache.get(key)
    if cached is not None:
        return cached

    # 2. Miss — try to become the "leader" who recomputes
    lock_key = f"lock:{key}"
    token = str(uuid.uuid4())

    for _ in range(100):  # bounded retry
        acquired = cache.set(lock_key, token, nx=True, px=lock_ttl_ms)
        if acquired:
            try:
                # 3. Double-check: maybe another worker populated while we waited
                cached = cache.get(key)
                if cached is not None:
                    return cached
                # 4. We're the leader — read DB and populate cache
                value = db.read(key)
                if value is not None:
                    cache.set(key, value, ex=60)
                return value
            finally:
                release_lock(keys=[lock_key], args=[token])
        else:
            # 5. Someone else is recomputing — wait briefly and retry the cache
            time.sleep(wait_ms / 1000)
            cached = cache.get(key)
            if cached is not None:
                return cached
            # cache still empty → loop and try to acquire ourselves
    raise RuntimeError("single_flight_get: gave up after too many retries")


run_stampede(single_flight_get, "single-flight with Redis lock")


With the lock, **exactly 1 DB read** happens no matter how many workers pile
in — the other 19 wait ~50ms and then read the freshly populated cache.
That's the whole trick.

### 💭 Other stampede mitigations (briefly)

- **Probabilistic early recomputation (XFetch):** refresh the key *before*
  it expires, with a probability that grows as TTL approaches 0. Avoids the
  moment-of-expiry cliff. See the XFetch paper for the math.
- **Stale-while-revalidate:** serve the *stale* value immediately and refresh
  in the background. Popular in HTTP caches and CDNs.
- **Request coalescing in-process:** Python's `asyncio` + a `dict` of in-flight
  futures is a lightweight version of single-flight for a single process.

For most Redis-backed services, the distributed lock above is the standard
answer.


---

## Part 6: Negative Caching — Cache the *Absence* of a Value

What happens when someone asks for `user:does-not-exist` a million times? Every
single call hits the DB, gets `None`, and returns. The cache never helps
because we only store *hits*.

**Negative caching** fixes this by storing a **sentinel value** that means
"this key really doesn't exist" — with a **shorter TTL** than normal entries,
so newly-created records show up quickly.

> ⚠️ Don't cache a bare Python `None` — you can't distinguish it from "not in
> cache at all". Use an explicit marker like `"__MISS__"`.


In [ ]:
MISS_SENTINEL = "__MISS__"
POSITIVE_TTL = 300     # 5 minutes for real values
NEGATIVE_TTL = 30      # 30 seconds for "not found" — short so new records appear quickly


def get_with_negative_cache(key: str):
    cached = cache.get(key)
    if cached == MISS_SENTINEL:
        return None           # we've recorded that this key doesn't exist
    if cached is not None:
        return cached         # real cached value
    # Miss — go to DB
    value = db.read(key)
    if value is None:
        cache.set(key, MISS_SENTINEL, ex=NEGATIVE_TTL)
    else:
        cache.set(key, value, ex=POSITIVE_TTL)
    return value


cache.flushall()
db.reset_counters()

# Ask for a key that doesn't exist — 5 times
for i in range(5):
    get_with_negative_cache("user:ghost")

print(f"DB reads for 5 lookups of a missing key: {db.read_count}  (should be 1)")


---

## 🌍 Real-World Examples

| System | Pattern | Notes |
|--------|---------|-------|
| **Facebook TAO / Memcached** | Cache-aside + lease tokens | Leases are Facebook's flavor of single-flight — prevents stampedes for hot celebrity profiles. |
| **Instagram feed** | Read-through + stale-while-revalidate | Feed service serves slightly stale data and refreshes asynchronously. |
| **Amazon DynamoDB DAX** | Write-through | DAX writes to cache and DynamoDB synchronously for transparency. |
| **Linux page cache** | Write-behind | Dirty pages flush to disk asynchronously — fast, but `sync` exists for a reason. |
| **Twitter timelines** | Negative caching | Profile lookups for deleted / non-existent accounts are cached to protect the user service. |

## 🧪 Try It Yourself

1. **Increase the worker count** to 200 in the stampede demo. The naive version
   will do ~200 DB reads; single-flight still does 1. See the DB latency drop.
2. **Kill the lock holder** mid-recompute (e.g. add `raise Exception()` inside
   the leader branch). Verify the lock still auto-expires so the system recovers.
3. **Break write-through** by failing the DB write (`raise` inside `db.write`
   temporarily). What ends up in the cache? How would you detect this?
4. **Combine patterns**: add a distributed lock to the read-through wrapper.
   Now you have a production-grade cache client in ~30 lines.

## 📝 Key Takeaways

| Pattern | Who owns DB logic | Fresh on write? | Risk |
|---------|------------------|-----------------|------|
| Cache-aside | App | Only with TTL + invalidate | Stale without invalidation |
| Read-through | Cache wrapper | Same as cache-aside | Same, just tidier API |
| Write-through | App → cache → DB (sync) | Always | Slower writes |
| Write-behind | App → cache (sync) → DB (async) | Eventually | **Data loss on crash** |

| Problem | Fix |
|---------|-----|
| Stale cache | TTL + invalidate-on-write |
| Hot-key stampede | Redis lock (SET NX EX) with token + Lua release |
| Misses hammering DB | Negative cache with sentinel + short TTL |

## ➡️ Next

You've now seen the full distributed-cache stack: partitioning (NB1), coherence (NB2),
consistent hashing (NB3), and access patterns + stampede protection (NB4). Together
these are what Redis Cluster, Memcached, DynamoDB DAX, and Facebook TAO all do
under the hood.


In [ ]:
# Cleanup
cache.flushall()
print("🧹 Cache flushed.")
